<a href="https://colab.research.google.com/github/almendraapolaya/DI_Bootcamp_a/blob/main/Week_7/Day_5/Exercises%20/Mini_project_w7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mini-project 1 : Brazillian E-commerce
===

**Project Overview**

In this mini-project, you will explore and analyze a Brazilian e-commerce dataset to uncover insights and trends. You will perform data loading, cleaning, and various SQL queries to extract meaningful information about the e-commerce platform’s operations and performance.

**1. Load and Explore the Data :**

In [ ]:
!unzip "Brazilian E-Commerce Dive Deep using SQL.zip"

In [ ]:
import numpy as np
import pandas as pd
import sqlite3
from sqlalchemy import create_engine

base_path = 'Brazilian E-Commerce Dive Deep using SQL/Brazilian E-Commerce Dive Deep using SQL/'

df_olist_customers = pd.read_csv(base_path + 'olist_customers_dataset.csv/olist_customers_dataset.csv')
df_olist_sellers = pd.read_csv(base_path + 'olist_sellers_dataset.csv')
df_olist_order_reviews = pd.read_csv(base_path + 'olist_order_reviews_dataset.csv/olist_order_reviews_dataset.csv')
df_olist_order_items = pd.read_csv(base_path + 'olist_order_items_dataset.csv/olist_order_items_dataset.csv')
df_olist_products = pd.read_csv(base_path + 'olist_products_dataset.csv/olist_products_dataset.csv')
df_olist_geolocation = pd.read_csv(base_path + 'olist_geolocation_dataset.csv/olist_geolocation_dataset.csv')
df_product_category_name_translation = pd.read_csv(base_path + 'product_category_name_translation.csv')
df_olist_orders = pd.read_csv(base_path + 'olist_orders_dataset.csv/olist_orders_dataset.csv')
df_olist_order_payments = pd.read_csv(base_path + 'olist_order_payments_dataset.csv/olist_order_payments_dataset.csv')

engine = create_engine('sqlite://', echo=False)

df_olist_customers.to_sql("olist_customers", con=engine, index=False)
df_olist_sellers.to_sql("olist_sellers", con=engine, index=False)
df_olist_order_reviews.to_sql("olist_order_reviews", con=engine, index=False)
df_olist_order_items.to_sql("olist_order_items", con=engine, index=False)
df_olist_products.to_sql("olist_products_dataset", con=engine, index=False)
df_olist_geolocation.to_sql("olist_geolocation", con=engine, index=False)
df_product_category_name_translation.to_sql("product_category_name_translation", con=engine, index=False)
df_olist_orders.to_sql("olist_orders", con=engine, index=False)
df_olist_order_payments.to_sql("olist_order_payments", con=engine, index=False)

print("Data loaded and SQL tables created successfully!")

In [ ]:
sql = '''
SELECT * FROM olist_customers
LIMIT 5
'''

df_sql = pd.read_sql_query(sql, con=engine)
df_sql

**2. Create SQLite Database and Export DataFrames:**

In [ ]:
from sqlalchemy import create_engine

engine = create_engine('sqlite:///brazilian_ecommerce.db', echo=False)

df_olist_customers.to_sql("olist_customers", con=engine, index=False, if_exists='replace')
df_olist_sellers.to_sql("olist_sellers", con=engine, index=False, if_exists='replace')
df_olist_order_reviews.to_sql("olist_order_reviews", con=engine, index=False, if_exists='replace')
df_olist_order_items.to_sql("olist_order_items", con=engine, index=False, if_exists='replace')
df_olist_products.to_sql("olist_products", con=engine, index=False, if_exists='replace')
df_olist_geolocation.to_sql("olist_geolocation", con=engine, index=False, if_exists='replace')
df_product_category_name_translation.to_sql("product_category_translation", con=engine, index=False, if_exists='replace')
df_olist_orders.to_sql("olist_orders", con=engine, index=False, if_exists='replace')
df_olist_order_payments.to_sql("olist_order_payments", con=engine, index=False, if_exists='replace')

print("Step 2 Complete: SQLite database created and tables populated!")

**3. Query 1: Count and Percentage of Orders Purchased in Jan 2018 with 5 Review Score:**

In [ ]:
sql_query_1 = '''
SELECT
    COUNT(*) AS total_january_orders,
    SUM(CASE WHEN review_score = 5 THEN 1 ELSE 0 END) AS five_star_orders,
    ROUND(AVG(CASE WHEN review_score = 5 THEN 100.0 ELSE 0.0 END), 2) AS percentage_five_star
FROM olist_orders o
JOIN olist_order_reviews r ON o.order_id = r.order_id
WHERE o.order_purchase_timestamp >= '2018-01-01'
  AND o.order_purchase_timestamp <= '2018-01-31 23:59:59';
'''

df_jan_reviews = pd.read_sql_query(sql_query_1, con=engine)
df_jan_reviews

**4. Query 2: Customer Purchase Trend Year-on-Year:**

In [ ]:
sql_query_2 = '''
SELECT
    strftime('%Y', order_purchase_timestamp) AS purchase_year,
    COUNT(order_id) AS total_orders
FROM olist_orders
GROUP BY purchase_year
ORDER BY purchase_year;
'''

df_yearly_trend = pd.read_sql_query(sql_query_2, con=engine)
df_yearly_trend

**5. Query 3: Average Order Values of Customers:**

In [ ]:
sql_query_3 = '''
SELECT
    AVG(order_total) AS average_order_value
FROM (
    SELECT
        order_id,
        SUM(payment_value) AS order_total
    FROM olist_order_payments
    GROUP BY order_id
) AS order_totals;
'''

df_aov = pd.read_sql_query(sql_query_3, con=engine)
df_aov

**6. Query 4: Top 5 Cities with Highest Revenue from 2016 to 2018:**

In [ ]:
sql_query_4 = '''
SELECT
    c.customer_city,
    SUM(p.payment_value) AS total_revenue
FROM olist_orders o
JOIN olist_order_payments p ON o.order_id = p.order_id
JOIN olist_customers c ON o.customer_id = c.customer_id
WHERE strftime('%Y', o.order_purchase_timestamp) BETWEEN '2016' AND '2018'
GROUP BY c.customer_city
ORDER BY total_revenue DESC
LIMIT 5;
'''

df_top_cities = pd.read_sql_query(sql_query_4, con=engine)
df_top_cities

**7. Query 5: State Wise Revenue Table Between 2016 to 2018:**

In [ ]:
sql_query_5 = '''
SELECT
    c.customer_state,
    SUM(p.payment_value) AS total_revenue
FROM olist_orders o
JOIN olist_order_payments p ON o.order_id = p.order_id
JOIN olist_customers c ON o.customer_id = c.customer_id
WHERE strftime('%Y', o.order_purchase_timestamp) BETWEEN '2016' AND '2018'
GROUP BY c.customer_state
ORDER BY total_revenue DESC;
'''

df_state_revenue = pd.read_sql_query(sql_query_5, con=engine)
df_state_revenue

**8. Query 6: Top Successful Sellers in Terms of Goods Sold, Revenue, and Customer Count:**

In [ ]:
sql_query_6 = '''
SELECT
    oi.seller_id,
    COUNT(oi.order_item_id) AS total_goods_sold,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    COUNT(DISTINCT o.customer_id) AS unique_customers,
    SUM(CASE WHEN r.review_score = 5 THEN 1 ELSE 0 END) AS five_star_ratings
FROM olist_order_items oi
JOIN olist_orders o ON oi.order_id = o.order_id
JOIN olist_order_reviews r ON oi.order_id = r.order_id
GROUP BY oi.seller_id
ORDER BY total_revenue DESC
LIMIT 10;
'''

df_top_sellers = pd.read_sql_query(sql_query_6, con=engine)
df_top_sellers

**9. Query 7: Delivery Success Rate Across States:**

In [ ]:
sql_query_7 = '''
SELECT
    c.customer_state,
    COUNT(o.order_id) AS total_orders,
    SUM(CASE WHEN o.order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
    ROUND(SUM(CASE WHEN o.order_status = 'delivered' THEN 1.0 ELSE 0.0 END) / COUNT(o.order_id) * 100, 2) AS success_rate_percentage
FROM olist_orders o
JOIN olist_customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY success_rate_percentage DESC;
'''

df_delivery_success = pd.read_sql_query(sql_query_7, con=engine)
df_delivery_success

**10. Query 8: Preferred Form of Payment for Different Categories:**

In [ ]:
sql_query_8 = '''
WITH CategoryPaymentCounts AS (
    SELECT
        p.product_category_name,
        pay.payment_type,
        COUNT(*) AS count_usage
    FROM olist_products p
    JOIN olist_order_items oi ON p.product_id = oi.product_id
    JOIN olist_order_payments pay ON oi.order_id = pay.order_id
    WHERE p.product_category_name IS NOT NULL
    GROUP BY p.product_category_name, pay.payment_type
),
RankedPayments AS (
    SELECT
        product_category_name,
        payment_type,
        count_usage,
        RANK() OVER (PARTITION BY product_category_name ORDER BY count_usage DESC) as rank
    FROM CategoryPaymentCounts
)
SELECT
    product_category_name,
    payment_type AS preferred_payment_method,
    count_usage
FROM RankedPayments
WHERE rank = 1
ORDER BY count_usage DESC;
'''

df_preferred_payment = pd.read_sql_query(sql_query_8, con=engine)
df_preferred_payment

**11. Query 9: Distance Between Cities:**

In [ ]:
import math
from sqlalchemy import text

def haversine(lat1, lon1, lat2, lon2):
    if None in (lat1, lon1, lat2, lon2):
        return None
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlamb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlamb/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return round(R * c, 2)

raw_conn = engine.raw_connection()
raw_conn.create_function("haversine", 4, haversine)

with engine.connect() as conn:
    conn.execute(text('''
    CREATE TABLE IF NOT EXISTS zip_coords AS
    SELECT
        geolocation_zip_code_prefix AS zip,
        AVG(geolocation_lat) AS lat,
        AVG(geolocation_lng) AS lng
    FROM olist_geolocation
    GROUP BY zip
    '''))
    conn.commit()

sql_query_9 = '''
SELECT
    o.order_id,
    c.customer_city,
    s.seller_city,
    haversine(gc.lat, gc.lng, gs.lat, gs.lng) AS distance_km
FROM olist_orders o
JOIN olist_customers c ON o.customer_id = c.customer_id
JOIN olist_order_items oi ON o.order_id = oi.order_id
JOIN olist_sellers s ON oi.seller_id = s.seller_id
JOIN zip_coords gc ON c.customer_zip_code_prefix = gc.zip
JOIN zip_coords gs ON s.seller_zip_code_prefix = gs.zip
LIMIT 10;
'''

df_distances = pd.read_sql_query(sql_query_9, con=raw_conn)
df_distances